In [1]:
%useLatestDescriptors
%use koog

In [9]:
val agentStrategy = strategy<String, String>("是否调用工具策略图") {
    val nodeSendInput by nodeLLMRequest()
    val nodeExecuteTool by nodeExecuteTool()
    val nodeSendToolResult by nodeLLMSendToolResult()

    edge(nodeStart forwardTo nodeSendInput)

    edge(
        (nodeSendInput forwardTo nodeFinish)
            .transformed { it }
            .onAssistantMessage { true }
    )

    edge(
        (nodeSendInput forwardTo nodeExecuteTool)
            .onToolCall { true }
    )

    edge(nodeExecuteTool forwardTo nodeSendToolResult)

    edge(
        (nodeSendToolResult forwardTo nodeFinish)
            .transformed { it }
            .onAssistantMessage { true }
    )

    edge(
        (nodeSendToolResult forwardTo nodeExecuteTool)
            .onToolCall { true }
    )
}

In [10]:
//    val mermaidDiagram: String = agentStrategy.asMermaidDiagram()

//    println(mermaidDiagram)

val mraidDiagram : String = agentStrategy.asMermaidDiagram()
println(mraidDiagram)

---
title: 是否调用工具策略图
---
stateDiagram
    state "nodeSendInput" as nodeSendInput
    state "nodeExecuteTool" as nodeExecuteTool
    state "nodeSendToolResult" as nodeSendToolResult

    [*] --> nodeSendInput
    nodeSendInput --> [*] : transformed
    nodeSendInput --> nodeExecuteTool : onCondition
    nodeExecuteTool --> nodeSendToolResult
    nodeSendToolResult --> [*] : transformed
    nodeSendToolResult --> nodeExecuteTool : onCondition


In [ ]:
println("")

In [7]:
import ai.koog.agents.core.environment.ReceivedToolResult

val myStrategy = strategy<String, String>("策略图"){
    val nodeSendInput by nodeLLMRequest() //
    val nodeExecutor by nodeExecuteTool()
    val nodeSendToolResult by nodeLLMSendToolResult()
    val compressHistory by nodeLLMCompressHistory<ReceivedToolResult>()
    edge(nodeStart forwardTo nodeSendInput)
    edge((nodeSendInput forwardTo nodeFinish).transformed{msg-> msg.content}.onAssistantMessage { true })

    edge(
        (nodeSendInput forwardTo nodeExecutor).onToolCall { true }
    )
    edge(
        (nodeExecutor forwardTo compressHistory)
            .onCondition { llm.readSession { prompt.messages.size > 10 } }
    )

    edge(
        (nodeExecutor forwardTo nodeSendToolResult)
    )
    edge(
        (nodeSendToolResult forwardTo nodeFinish).transformed { it }.onAssistantMessage { true }
    )
    edge(
        (nodeSendToolResult forwardTo nodeExecutor).onToolCall { true }
    )
}

val  mraidDiagram : String = myStrategy.asMermaidDiagram()
println(mraidDiagram)


---
title: 策略图
---
stateDiagram
    state "nodeSendInput" as nodeSendInput
    state "nodeExecutor" as nodeExecutor
    state "compressHistory" as compressHistory
    state "nodeSendToolResult" as nodeSendToolResult

    [*] --> nodeSendInput
    nodeSendInput --> [*] : transformed
    nodeSendInput --> nodeExecutor : onCondition
    nodeExecutor --> compressHistory : onCondition
    nodeExecutor --> nodeSendToolResult
    nodeSendToolResult --> [*] : transformed
    nodeSendToolResult --> nodeExecutor : onCondition


In [8]:
// Define that the history is too long if there are more than 100 messages
//private suspend fun AIAgentContext.historyIsTooLong(): Boolean = llm.readSession { prompt.messages.size > 100 }

val strategy = strategy<String, String>("execute-with-history-compression") {
    val callLLM by nodeLLMRequest()
    val executeTool by nodeExecuteTool()
    val sendToolResult by nodeLLMSendToolResult()

    // Compress the LLM history and keep the current ReceivedToolResult for the next node
    val compressHistory by nodeLLMCompressHistory<ReceivedToolResult>()

    edge(nodeStart forwardTo callLLM)
    edge(callLLM forwardTo nodeFinish onAssistantMessage { true })
    edge(callLLM forwardTo executeTool onToolCall { true })

    // Compress history after executing any tool if the history is too long
    edge((executeTool forwardTo compressHistory).onCondition { llm.readSession { prompt.messages.size > 10 } } )
    edge(compressHistory forwardTo sendToolResult)
    // Otherwise, proceed to the next LLM request
    edge(executeTool forwardTo sendToolResult onCondition { !llm.readSession { prompt.messages.size > 10 } })

    edge(sendToolResult forwardTo executeTool onToolCall { true })
    edge(sendToolResult forwardTo nodeFinish onAssistantMessage { true })
}

val  Mystrategy : String = strategy.asMermaidDiagram()
println(Mystrategy)

---
title: execute-with-history-compression
---
stateDiagram
    state "callLLM" as callLLM
    state "executeTool" as executeTool
    state "compressHistory" as compressHistory
    state "sendToolResult" as sendToolResult

    [*] --> callLLM
    callLLM --> [*] : transformed
    callLLM --> executeTool : onCondition
    executeTool --> compressHistory : onCondition
    executeTool --> sendToolResult : onCondition
    compressHistory --> sendToolResult
    sendToolResult --> executeTool : onCondition
    sendToolResult --> [*] : transformed


In [13]:
val agentStrategy = strategy<String, String>("是否调用工具策略图") {
    val nodeSendInput by nodeLLMRequest()
    val nodeExecuteTool by nodeExecuteTool()
    val nodeSendToolResult by nodeLLMSendToolResult()
    val compressHistory by nodeLLMCompressHistory<ReceivedToolResult>()

    edge(nodeStart forwardTo nodeSendInput)

    edge(
        (nodeSendInput forwardTo nodeFinish)
            .transformed { it }
            .onAssistantMessage { true }
    )

    edge(
        (nodeSendInput forwardTo nodeExecuteTool)
            .onToolCall { true }
    )
    edge(
        (nodeExecuteTool forwardTo compressHistory)
            .onCondition { llm.readSession { prompt.messages.size >10 } }
    )

    edge(
        (nodeExecuteTool forwardTo nodeSendToolResult)
            .onCondition { !llm.readSession { prompt.messages.size >10 } }
    )

    edge(
        (compressHistory forwardTo nodeSendToolResult)

    )

//    edge(nodeExecuteTool forwardTo nodeSendToolResult)

    edge(
        (nodeSendToolResult forwardTo nodeFinish)
            .transformed { it }
            .onAssistantMessage { true }
    )

    edge(
        (nodeSendToolResult forwardTo nodeExecuteTool)
            .onToolCall { true }
    )
}

val  Mystrategy : String = agentStrategy.asMermaidDiagram()
println(Mystrategy)

---
title: 是否调用工具策略图
---
stateDiagram
    state "nodeSendInput" as nodeSendInput
    state "nodeExecuteTool" as nodeExecuteTool
    state "compressHistory" as compressHistory
    state "nodeSendToolResult" as nodeSendToolResult

    [*] --> nodeSendInput
    nodeSendInput --> [*] : transformed
    nodeSendInput --> nodeExecuteTool : onCondition
    nodeExecuteTool --> compressHistory : onCondition
    nodeExecuteTool --> nodeSendToolResult : onCondition
    compressHistory --> nodeSendToolResult
    nodeSendToolResult --> [*] : transformed
    nodeSendToolResult --> nodeExecuteTool : onCondition


In [2]:
import ai.koog.prompt.streaming.collectText

//private fun streamTextStrategy(strategyName: String) =
//
//    }

val streamStrategy= strategy<String, String>("流式输出") {
    // 预定义流式LLM节点，输出为 Flow<StreamFrame> 类型
    val llmNode by nodeLLMRequestStreaming("streaming-llm-node")
    // 起点 → 流式LLM节点
    edge(nodeStart forwardTo llmNode)
    // 流式节点输出转文本后 → 终点（collectText() 是Flow<StreamFrame>的扩展函数，拼接所有文本增量）
    edge(llmNode forwardTo nodeFinish transformed { it.collectText() })
}
val  Mystrategy : String = streamStrategy.asMermaidDiagram()
println(Mystrategy)



---
title: 流式输出
---
stateDiagram
    state "streaming-llm-node" as streaming_llm_node

    [*] --> streaming_llm_node
    streaming_llm_node --> [*] : transformed


```mermaid
---
title: 流式输出
---
stateDiagram
    state "streaming-llm-node" as streaming_llm_node

    [*] --> streaming_llm_node
    streaming_llm_node --> [*] : transformed
```



In [11]:
import ai.koog.agents.core.environment.ReceivedToolResult

val agentStrategy = strategy<String, String>("是否调用工具策略图") {
    val nodeSendInput by nodeLLMRequest()
    val nodeExecuteTool by nodeExecuteTool()
    val nodeSendToolResult by nodeLLMSendToolResult()
    val compressHistory by nodeLLMCompressHistory<ReceivedToolResult>()
    val llmNode by nodeLLMRequestStreaming("streaming-llm-node") // 用于流式输出

    edge(nodeStart forwardTo nodeSendInput)

    edge(
        ( nodeSendInput forwardTo llmNode)
            .transformed { it }
            .onAssistantMessage { true }
    )

    edge(
        (nodeSendInput forwardTo nodeExecuteTool)
            .onToolCall { true }
    )
    edge(
        (nodeExecuteTool forwardTo compressHistory)
            .onCondition { llm.readSession { prompt.messages.size >10 } }
    )

    edge(
        (nodeExecuteTool forwardTo nodeSendToolResult)
            .onCondition { !llm.readSession { prompt.messages.size >10 } }
    )

    edge(
        (compressHistory forwardTo nodeSendToolResult)

    )

    edge(
        (nodeSendToolResult forwardTo llmNode)
            .transformed { it }
            .onAssistantMessage { true }
    )



    edge(
        (llmNode forwardTo nodeFinish)
            .onToolCall { true }
            .onAssistantMessage { true }
    )

    edge(
        (nodeSendToolResult forwardTo nodeExecuteTool)
            .onToolCall { true }
    )
}

val StreamStrategy : String = agentStrategy.asMermaidDiagram()
println(StreamStrategy)

---
title: 是否调用工具策略图
---
stateDiagram
    state "nodeSendInput" as nodeSendInput
    state "streaming-llm-node" as streaming_llm_node
    state "nodeExecuteTool" as nodeExecuteTool
    state "compressHistory" as compressHistory
    state "nodeSendToolResult" as nodeSendToolResult

    [*] --> nodeSendInput
    nodeSendInput --> streaming_llm_node : transformed
    nodeSendInput --> nodeExecuteTool : onCondition
    streaming_llm_node --> [*] : transformed
    nodeExecuteTool --> compressHistory : onCondition
    nodeExecuteTool --> nodeSendToolResult : onCondition
    compressHistory --> nodeSendToolResult
    nodeSendToolResult --> streaming_llm_node : transformed
    nodeSendToolResult --> nodeExecuteTool : onCondition


```mermaid
---
title: 是否调用工具策略图
---
stateDiagram
    state "nodeSendInput" as nodeSendInput
    state "streaming-llm-node" as streaming_llm_node
    state "nodeExecuteTool" as nodeExecuteTool
    state "compressHistory" as compressHistory
    state "nodeSendToolResult" as nodeSendToolResult

    [*] --> nodeSendInput
    nodeSendInput --> streaming_llm_node : transformed
    nodeSendInput --> nodeExecuteTool : onCondition
    streaming_llm_node --> [*] : transformed
    nodeExecuteTool --> compressHistory : onCondition
    nodeExecuteTool --> nodeSendToolResult : onCondition
    compressHistory --> nodeSendToolResult
    nodeSendToolResult --> streaming_llm_node : transformed
    nodeSendToolResult --> nodeExecuteTool : onCondition

```

In [15]:
val agentStrategy = strategy<String, String>("是否调用工具策略图") {
    val nodeSendInput by nodeLLMRequest()
    val nodeExecuteTool by nodeExecuteTool()
    val nodeSendToolResult by nodeLLMSendToolResult()
    val compressHistory by nodeLLMCompressHistory<ReceivedToolResult>()
    val llmNode by nodeLLMRequestStreaming("streaming-llm-node") //

    edge(nodeStart forwardTo nodeSendInput)

    edge(
        (nodeSendInput forwardTo nodeFinish)
            .transformed { it }
            .onAssistantMessage { true }
    )

    edge(
        (nodeSendInput forwardTo nodeExecuteTool)
            .onToolCall { true }
    )
    edge(
        (nodeExecuteTool forwardTo compressHistory)
            .onCondition { llm.readSession { prompt.messages.size >10 } }
    )

    edge(
        (nodeExecuteTool forwardTo nodeSendToolResult)
            .onCondition { !llm.readSession { prompt.messages.size >10 } }
    )

    edge(
        (compressHistory forwardTo nodeSendToolResult)

    )


    edge(
        (nodeSendToolResult forwardTo nodeFinish)
            .transformed { it }
            .onAssistantMessage { true }
    )

    edge(
        (nodeSendToolResult forwardTo nodeExecuteTool)
            .onToolCall { true }
    )
}
val StreamStrategy : String = agentStrategy.asMermaidDiagram()
println(StreamStrategy)

---
title: 是否调用工具策略图
---
stateDiagram
    state "nodeSendInput" as nodeSendInput
    state "nodeExecuteTool" as nodeExecuteTool
    state "compressHistory" as compressHistory
    state "nodeSendToolResult" as nodeSendToolResult

    [*] --> nodeSendInput
    nodeSendInput --> [*] : transformed
    nodeSendInput --> nodeExecuteTool : onCondition
    nodeExecuteTool --> compressHistory : onCondition
    nodeExecuteTool --> nodeSendToolResult : onCondition
    compressHistory --> nodeSendToolResult
    nodeSendToolResult --> [*] : transformed
    nodeSendToolResult --> nodeExecuteTool : onCondition


```mermaid
---
title: 是否调用工具策略图
---
stateDiagram
    state "nodeSendInput" as nodeSendInput
    state "nodeExecuteTool" as nodeExecuteTool
    state "compressHistory" as compressHistory
    state "nodeSendToolResult" as nodeSendToolResult

    [*] --> nodeSendInput
    nodeSendInput --> [*] : transformed
    nodeSendInput --> nodeExecuteTool : onCondition
    nodeExecuteTool --> compressHistory : onCondition
    nodeExecuteTool --> nodeSendToolResult : onCondition
    compressHistory --> nodeSendToolResult
    nodeSendToolResult --> [*] : transformed
    nodeSendToolResult --> nodeExecuteTool : onCondition

```

In [ ]:
val agentStrategy = strategy<String, String>("是否调用工具策略图-流式") {
    // 流式LLM节点，统一处理工具调用和文本输出
    val llmNode by nodeLLMRequestStreaming("streaming-llm-node")
    val nodeExecuteTool by nodeExecuteTool()
    val nodeSendToolResult by nodeLLMSendToolResult()
    val compressHistory by nodeLLMCompressHistory<ReceivedToolResult>()

    edge(nodeStart forwardTo llmNode)

    // 处理流式输出：收集流，判断是工具调用还是文本
    edge(llmNode forwardTo nodeExecuteTool transformed { flow ->
        // 提取流中的工具调用信息，传给工具执行节点
        val toolCallFrames = flow.filterIsInstance<StreamFrame.ToolCallDelta>().toList()
        // 实际需要根据你的工具格式转成Message.Tool.Call，这里仅示例
        // 若为文本则直接返回给终点
        flow.collectText()
    })

    // 原有工具执行、历史压缩逻辑不变，注意nodeSendToolResult的输出是Message，可以连回llmNode（需要String输入，需提取Message内容）
    edge(nodeExecuteTool forwardTo compressHistory onCondition { llm.readSession { prompt.messages.size > 10 } })
    edge(nodeExecuteTool forwardTo nodeSendToolResult onCondition { !llm.readSession { prompt.messages.size > 10 } })
    edge(compressHistory forwardTo nodeSendToolResult)

    // 工具结果返回后，连回流式节点重新调用LLM
    edge(nodeSendToolResult forwardTo llmNode transformed { (it as Message.Assistant).content })
    // 流式节点最终输出到终点，转成String
    edge(llmNode forwardTo nodeFinish transformed { it.collectText() })
}
